# Module 1: Finding Your Way Around AmsterdamUMCdb
## ESICM LIVES26 — Datathon Educational Series

---

> **This notebook does NOT cover basic setup or introductory SQL.**  
> The [official AmsterdamUMCdb getting started notebook](https://github.com/AmsterdamUMC/AmsterdamUMCdb/blob/master/bigquery/getting_started_v2.ipynb) handles that. Start there if you haven't already.  
>  
> **It covers what you discover *after* you get started** — the process, the gotchas, and the things that aren't in the documentation.

> **⚠ Datathon context disclaimer:**  
> This notebook documents how we approached problems during the **2026 ESICM INDICATE datathon**. The database evolves between versions and datathon instances — some issues described here may already be resolved in your instance, and new ones may have appeared.  
>  
> Use this to think through how to investigate and find what you need — not as a definitive reference for what you will find. When something doesn't work, ask: *is this a me problem, a query problem, or a database problem?* The techniques here help you answer that.

---

### Learning Objectives

By the end of this module, you will be able to:

1. **Navigate** the OMOP CDM table structure in BigQuery and identify which tables contain the data you need
2. **Apply** a systematic 4-step process to discover and validate concept IDs
3. **Recognise** common datathon pitfalls (unit errors, concept synonyms, cumulative artefacts, visit linkage gaps, ventilation episode detection) and apply worked solutions

> *Bloom's taxonomy level: Knowledge — building the foundational skills needed before cohort construction.*

### What You'll Learn

| Section | Topic |
|---------|-------|
| 0 | Setup — two undocumented pitfalls that cause silent failures |
| 1 | Orient yourself — what tables actually exist in your instance |
| 2 | How to find and validate concept IDs — a 4-step process |
| 3 | Pitfalls & Pearls from real datathon experience |

**Everything here is drawn from the 2026 ESICM INDICATE datathon.** These are things that cost time. Hopefully not yours.

---
## Section 0: Setup

Two things to configure. Both have pitfalls that cause confusing errors — flagged below.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Replace these with your datathon project details

PROJECT_ID        = 'your-datathon-project'     # Your personal / team BigQuery project
DATASET_PROJECT_ID = 'amsterdamumcdb'            # AmsterdamUMCdb project — do not change
DATASET_ID        = 'van_gogh_2026_datathon'     # Dataset name — check your datathon materials
LOCATION          = 'eu'                         # ⚠ Must be 'eu' — AmsterdamUMCdb is EU-hosted

### ⚠ Pitfall: Two Separate BigQuery Configs Are Required

The BigQuery Python client needs **two separate config objects**. Most documentation shows only one. Using the wrong one causes a cryptic error:

```
BadRequest: 400 Cannot explicitly modify anonymous table
```

Here's the difference:

| Config | Purpose | When to use |
|--------|---------|-------------|
| `job_config` | Clean read-only config | Simple SELECT queries |
| `def_config` | Sets default dataset | Passed to client constructor — applies to all queries by default |

Use `job_config=job_config` explicitly in your `client.query()` calls for read queries. The `def_config` is wired into the client at construction — it sets the default dataset so you don't have to fully qualify every table name.

### ⚠ Pitfall: Location Must Be `'eu'`

AmsterdamUMCdb is hosted in the EU BigQuery region. If `LOCATION` is wrong or absent, queries either fail silently or return a region mismatch error. Set it explicitly every time.

In [ ]:
import os
import pandas as pd
import numpy as np
from google.colab import auth
from google.cloud import bigquery

# Authenticate to Google Cloud
auth.authenticate_user()
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID

# ⚠ Two separate configs required — see note above
job_config = bigquery.job.QueryJobConfig()

def_config = bigquery.job.QueryJobConfig(
    default_dataset=f'{DATASET_PROJECT_ID}.{DATASET_ID}'
)

client = bigquery.Client(
    project=PROJECT_ID,
    location=LOCATION,                   # ⚠ 'eu' required
    default_query_job_config=def_config
)

print(f'✓ Connected to {DATASET_PROJECT_ID}.{DATASET_ID}')
print(f'  Project: {PROJECT_ID} | Location: {LOCATION}')

### 💡 Pearl: Increase Colab Data Table Limits Early

The default Colab interactive data table cuts off at ~20 rows and ~20 columns — far too small for any real exploration. Set these before you start querying.

In [ ]:
%load_ext google.colab.data_table
from google.colab.data_table import DataTable

DataTable.max_columns = 50
DataTable.max_rows    = 80000   # Default is far too small for exploration

print('✓ Data table limits set')

---
## Section 1: Orient Yourself First — What Tables Exist?

Before writing any cohort queries, find out what you have access to. **Table availability varies between datathon instances and the public dataset.** This takes 30 seconds and saves hours of debugging queries against tables that don't exist.

Run this on day one of any datathon.

In [ ]:
query = f"""
SELECT table_name
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
"""

tables = client.query(query, job_config=job_config).to_dataframe()
print(f'Tables available in {DATASET_ID}:')
print(tables.to_string(index=False))

### ⚠ Pitfall: `procedure_occurrence` May Be Empty

In the 2026 datathon, `procedure_occurrence` existed but was **not reliably populated**. If you plan to identify ventilation, procedures, or clinical interventions from procedure codes — verify the table has data *before* building logic on it.

If it returns zero rows: you need a **measurement-based approach** instead (see Section 3, Pitfall 1).

In [ ]:
query = f"""
SELECT
  COUNT(*)               AS row_count,
  COUNT(DISTINCT person_id) AS patient_count
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.procedure_occurrence`
"""

result = client.query(query, job_config=job_config).to_dataframe()
print('procedure_occurrence contents:')
print(result)

if result['row_count'].iloc[0] == 0:
    print('\n⚠ Table is empty — use measurement-based approach instead')
else:
    print(f"\n✓ Table has data — {result['row_count'].iloc[0]:,} procedures across {result['patient_count'].iloc[0]:,} patients")

### 💡 Pattern: Always Inspect Columns Before Querying a New Table

OMOP column names vary by implementation. Before assuming a standard field exists, check what's actually there. One minute now saves an hour of debugging later.

The helper below works for any table — reuse it throughout the datathon.

In [ ]:
def inspect_table(table_name):
    """Return column names and data types for any table."""
    query = f"""
    SELECT column_name, data_type
    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name = '{table_name}'
    ORDER BY ordinal_position
    """
    return client.query(query, job_config=job_config).to_dataframe()

# Inspect measurement — the most important table for ICU research
measurement_cols = inspect_table('measurement')
print('measurement table columns:')
print(measurement_cols.to_string(index=False))

# Tip: also try inspect_table('visit_occurrence'), inspect_table('drug_exposure'), etc.

---
## Section 2: How to Find Concept IDs — The 4-Step Process

Concept IDs are the currency of OMOP. They identify every measured variable, drug, procedure, and observation.

> **Important:** Concept IDs cited in published papers or examples are dataset-specific. Always derive and validate your own for the datathon instance you're working with. The 4-step process below works every time.

| Step | What you do |
|------|-------------|
| 1 | Search concept table by name — cast wide |
| 2 | Cross-reference against actual measurement data |
| 3 | Plausibility spot-check with real patient values |
| 4 | Hunt for synonyms — the same variable often has multiple IDs |

### Step 1: Search the Concept Table by Name

Start broad. Use `LIKE '%term%'` searches. You'll get more results than you need — that's fine. You'll narrow down in Step 2.

Try multiple terms: e.g. for oxygen: `'fio2'`, `'oxygen'`, `'inspired'`.

In [ ]:
# ← Change this to the variable you're looking for
search_term = 'ventilat'

query = f"""
SELECT
  concept_id,
  concept_name,
  domain_id,
  vocabulary_id,
  concept_class_id,
  standard_concept
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.concept`
WHERE LOWER(concept_name) LIKE '%{search_term}%'
ORDER BY concept_name
LIMIT 100
"""

concepts = client.query(query, job_config=job_config).to_dataframe()
print(f"Found {len(concepts)} concepts matching '{search_term}'")
concepts

### Step 2: Cross-Reference Against What's Actually Measured

Being in the concept table **doesn't mean the variable has data**. Join against the measurement table to see real counts.

> **💡 Pearl:** Always look at both `measurement_count` AND `patient_count` together.  
> - High measurement count + low patient count → frequent in few patients → unlikely cohort-defining  
> - Sort by `measurement_count DESC` — the top variables account for the vast majority of recorded data

Paste your candidate IDs from Step 1 into `candidate_ids` below.

In [ ]:
# ← Paste your candidate concept IDs from Step 1
candidate_ids = []  # e.g. [2000000204, 3022875, 21490752]

if candidate_ids:
    ids_str = ','.join(str(i) for i in candidate_ids)
    query = f"""
    SELECT
      m.measurement_concept_id,
      c.concept_name,
      c.vocabulary_id,
      COUNT(*)                   AS measurement_count,
      COUNT(DISTINCT m.person_id) AS patient_count,
      MIN(m.value_as_number)     AS min_value,
      MAX(m.value_as_number)     AS max_value,
      ROUND(AVG(m.value_as_number), 2) AS mean_value
    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
    JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
      ON m.measurement_concept_id = c.concept_id
    WHERE m.measurement_concept_id IN ({ids_str})
    GROUP BY m.measurement_concept_id, c.concept_name, c.vocabulary_id
    ORDER BY measurement_count DESC
    """

    cross_ref = client.query(query, job_config=job_config).to_dataframe()
    cross_ref
else:
    print('Set candidate_ids above to run this step')

In [ ]:
# Alternative: see ALL top measured variables — useful when you don't know what to look for yet

query = f"""
SELECT
  m.measurement_concept_id,
  c.concept_name,
  COUNT(*)                   AS measurement_count,
  COUNT(DISTINCT m.person_id) AS patient_count
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
  ON m.measurement_concept_id = c.concept_id
GROUP BY m.measurement_concept_id, c.concept_name
ORDER BY measurement_count DESC
LIMIT 50
"""

top_measurements = client.query(query, job_config=job_config).to_dataframe()
print('Top 50 most frequently measured variables:')
top_measurements

### Step 3: Plausibility Spot-Check

Before building cohort logic on a concept ID, sample 10 patients and look at the actual values. Are they physiologically plausible? Are there obvious unit errors or outliers?

**Fix this now.** Finding a unit problem after you've built a 500-line CTE is painful.

In [ ]:
# ← Replace with the concept ID you're validating
CONCEPT_ID = None

if CONCEPT_ID:
    query = f"""
    SELECT
      m.person_id,
      m.measurement_datetime,
      c.concept_name,
      m.value_as_number,
      m.unit_source_value
    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
    JOIN (
      SELECT DISTINCT person_id
      FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
      WHERE measurement_concept_id = {CONCEPT_ID}
      LIMIT 10
    ) s ON m.person_id = s.person_id
    JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
      ON m.measurement_concept_id = c.concept_id
    WHERE m.measurement_concept_id = {CONCEPT_ID}
    ORDER BY m.person_id, m.measurement_datetime
    LIMIT 200
    """

    spot_check = client.query(query, job_config=job_config).to_dataframe()
    print(f'Value distribution for concept {CONCEPT_ID}:')
    print(spot_check['value_as_number'].describe())
    print(f"\nUnits seen: {spot_check['unit_source_value'].unique()}")
    print('\nSample rows:')
    spot_check.head(20)
else:
    print('Set CONCEPT_ID above to run this check')

### Step 4: Check for Synonyms and Duplicate Concept IDs

The same physiological variable often appears under **multiple concept IDs** — different vocabularies, different source systems. If you pick just one ID, you may miss a significant fraction of the data.

Find all of them, then use `IN (id1, id2, ...)` in your cohort queries to capture everything.

> **💡 Pearl:** After running this, collect all relevant IDs into a list and use  
> `WHERE measurement_concept_id IN (id1, id2, id3)`  
> Do not pick just one and assume it captures everything.

In [ ]:
# ← Change to the variable you're searching for
synonym_term = 'oxygen saturation'

query = f"""
SELECT
  m.measurement_concept_id,
  c.concept_name,
  c.vocabulary_id,
  COUNT(*)                   AS measurement_count,
  COUNT(DISTINCT m.person_id) AS patient_count
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
  ON m.measurement_concept_id = c.concept_id
WHERE LOWER(c.concept_name) LIKE '%{synonym_term}%'
GROUP BY m.measurement_concept_id, c.concept_name, c.vocabulary_id
ORDER BY measurement_count DESC
"""

synonyms = client.query(query, job_config=job_config).to_dataframe()
print(f"All concept IDs for '{synonym_term}':")
synonyms

---
## Section 3: Pitfalls & Pearls

These are things discovered during the 2026 ESICM INDICATE datathon that are **not in the documentation**. Each cost time. Hopefully not yours.

### ⚠ Pitfall 1: One Concept ID Is Not Enough to Identify a Clinical State

**A single concept ID rarely defines a clinical condition on its own.**

Example — identifying invasive mechanical ventilation:
- **FiO2 alone ≠ invasive ventilation** — also present during NIV and high-flow oxygen therapy
- **PEEP alone ≠ invasive ventilation** — also present during CPAP
- **TV alone** — multiple concept IDs exist (set vs. expired); values overlap with NIV

You need to require **ALL THREE simultaneously**: FiO2 + PEEP + a parameter only present during invasive ventilation (e.g. set tidal volume). The logic: if a patient has all three being measured at the same time, they are almost certainly invasively ventilated.

**The same logic applies to any clinical state.** Ask: what combination of measurements is specific enough? Then validate against a clinical sample.

In [ ]:
# Pattern: requiring co-occurrence of multiple concept IDs to define a clinical state
# Run Section 2 first to find and validate your concept IDs

FIO2_ID   = None   # ← replace with your validated FiO2 concept ID
PEEP_ID   = None   # ← replace with your validated PEEP concept ID
TV_SET_ID = None   # ← replace with your validated set tidal volume concept ID

if all([FIO2_ID, PEEP_ID, TV_SET_ID]):
    query = f"""
    -- Only patients who have ALL THREE measured — not just one or two
    SELECT person_id
    FROM (
      SELECT
        person_id,
        MAX(CASE WHEN measurement_concept_id = {FIO2_ID}   THEN 1 ELSE 0 END) AS has_fio2,
        MAX(CASE WHEN measurement_concept_id = {PEEP_ID}   THEN 1 ELSE 0 END) AS has_peep,
        MAX(CASE WHEN measurement_concept_id = {TV_SET_ID} THEN 1 ELSE 0 END) AS has_tv
      FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
      WHERE measurement_concept_id IN ({FIO2_ID}, {PEEP_ID}, {TV_SET_ID})
      GROUP BY person_id
    )
    WHERE has_fio2 = 1 AND has_peep = 1 AND has_tv = 1
    """

    invasive_patients = client.query(query, job_config=job_config).to_dataframe()
    print(f'Patients with all three invasive ventilation markers: {len(invasive_patients):,}')
    print('\nCompare this against the count you get with FiO2 alone:')

    # Count with FiO2 alone for comparison
    query_fio2_only = f"""
    SELECT COUNT(DISTINCT person_id) AS fio2_only_count
    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
    WHERE measurement_concept_id = {FIO2_ID}
    """
    fio2_only = client.query(query_fio2_only, job_config=job_config).to_dataframe()
    print(f"  FiO2 alone:     {fio2_only['fio2_only_count'].iloc[0]:,} patients")
    print(f'  All three:      {len(invasive_patients):,} patients')
    print('  The difference represents patients on NIV/CPAP/high-flow — not invasively ventilated')
else:
    print('Set FIO2_ID, PEEP_ID, and TV_SET_ID above to run this query')
    print('Run Section 2 first to find and validate your concept IDs')

### ⚠ Pitfall 2: No Pre-Calculated Fluid Balances Exist

We searched across `measurement`, `observation`, and `drug_exposure` — there are **no pre-computed daily or cumulative fluid balance fields** in AmsterdamUMCdb. You must calculate fluid balance yourself from intake and output concept IDs.

> **💡 Pearl:** Search across ALL three tables when hunting for any variable. Fluid infusions may be in `drug_exposure`, not `measurement`.

The query below searches all three systematically.

In [ ]:
print('=== 1. Searching measurement table ===')
query_meas = f"""
SELECT
  c.concept_id,
  c.concept_name,
  COUNT(DISTINCT m.person_id) AS patient_count,
  COUNT(*) AS measurement_count
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
  ON m.measurement_concept_id = c.concept_id
WHERE LOWER(c.concept_name) LIKE '%fluid%'
   OR LOWER(c.concept_name) LIKE '%intake%'
   OR LOWER(c.concept_name) LIKE '%output%'
   OR LOWER(c.concept_name) LIKE '%urine%'
   OR LOWER(c.concept_name) LIKE '%balance%'
GROUP BY c.concept_id, c.concept_name
HAVING patient_count > 100
ORDER BY patient_count DESC
LIMIT 20
"""
fluid_meas = client.query(query_meas, job_config=job_config).to_dataframe()
print(fluid_meas.to_string(index=False) if len(fluid_meas) > 0 else '  No results')

print('\n=== 2. Searching drug_exposure table (infusions often live here) ===')
query_drug = f"""
SELECT
  c.concept_id,
  c.concept_name,
  COUNT(DISTINCT de.person_id) AS patient_count
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.drug_exposure` de
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
  ON de.drug_concept_id = c.concept_id
WHERE LOWER(c.concept_name) LIKE '%infusion%'
   OR LOWER(c.concept_name) LIKE '%crystalloid%'
   OR LOWER(c.concept_name) LIKE '%saline%'
   OR LOWER(c.concept_name) LIKE '%ringer%'
   OR LOWER(c.concept_name) LIKE '%colloid%'
GROUP BY c.concept_id, c.concept_name
HAVING patient_count > 100
ORDER BY patient_count DESC
LIMIT 20
"""
fluid_drug = client.query(query_drug, job_config=job_config).to_dataframe()
print(fluid_drug.to_string(index=False) if len(fluid_drug) > 0 else '  No results')

print('\n=== 3. Searching observation table ===')
query_obs = f"""
SELECT
  c.concept_id,
  c.concept_name,
  COUNT(DISTINCT o.person_id) AS patient_count
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.observation` o
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
  ON o.observation_concept_id = c.concept_id
WHERE LOWER(c.concept_name) LIKE '%fluid%'
   OR LOWER(c.concept_name) LIKE '%balance%'
GROUP BY c.concept_id, c.concept_name
HAVING patient_count > 100
ORDER BY patient_count DESC
LIMIT 20
"""
fluid_obs = client.query(query_obs, job_config=job_config).to_dataframe()
print(fluid_obs.to_string(index=False) if len(fluid_obs) > 0 else '  No results — no pre-calculated fluid balance found')

### ⚠ Pitfall 3: Dialysis Values Are Cumulative Totals, Not Incremental

If patients are on CRRT or intermittent dialysis, the **ultrafiltration measurement is stored as a running cumulative total** that resets periodically (e.g. at shift change or bag change). Using it directly will massively overstate fluid output.

You need to convert cumulative → incremental using `.diff()`, then handle reset points (where the cumulative value drops back toward zero).

The cell below shows the pattern with a simulated example.

In [ ]:
import pandas as pd

# Simulated cumulative dialysis values — resets at measurement 4
example_values = [0, 150, 320, 480, 100, 290, 450, 610]

df = pd.DataFrame({
    'measurement_number': range(len(example_values)),
    'cumulative_value':   example_values
})

# Convert cumulative to incremental
df['incremental'] = df['cumulative_value'].diff()

# Handle reset points: where cumulative drops (new bag started)
is_reset = (df['incremental'] < 0) | (df['incremental'].isna())
df.loc[is_reset, 'incremental'] = df.loc[is_reset, 'cumulative_value']

print('Cumulative → Incremental dialysis conversion:')
print(df.to_string(index=False))
print(f"\nTotal removed (last cumulative):   {df['cumulative_value'].iloc[-1]:.0f} mL")
print(f"Total removed (incremental sum):   {df['incremental'].sum():.0f} mL")
print('These should match — if they differ significantly, check for multiple reset points')
print('\n⚠ Apply this pattern to your dialysis DataFrame before calculating fluid balance')

### ⚠ Pitfall 4: `visit_occurrence_id` Is Not Reliably Populated in the Measurement Table

You cannot reliably use `visit_occurrence_id` in the measurement table to link measurements to admissions. In some implementations this field is sparsely populated or NULL for most rows.

**Verify before building on it.** If it's unreliable, use a time-based join to `visit_occurrence` instead:

```sql
JOIN visit_occurrence v
  ON m.person_id = v.person_id
  AND m.measurement_datetime BETWEEN v.visit_start_datetime AND v.visit_end_datetime
```

In [ ]:
# Check how well-populated visit_occurrence_id is in measurement
query = f"""
SELECT
  COUNT(*)                      AS total_rows,
  COUNT(visit_occurrence_id)    AS rows_with_visit_id,
  ROUND(
    COUNT(visit_occurrence_id) * 100.0 / COUNT(*), 1
  )                             AS pct_populated
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
LIMIT 1
"""

visit_check = client.query(query, job_config=job_config).to_dataframe()
print('visit_occurrence_id population rate in measurement table:')
print(visit_check)

pct = visit_check['pct_populated'].iloc[0]
if pct < 90:
    print(f'\n⚠ Only {pct}% populated — use time-based join to visit_occurrence instead')
else:
    print(f'\n✓ {pct}% populated — visit_occurrence_id linkage is reliable in this instance')

### ✅ Pitfall 5 (Resolved): Multiple ICU Admissions and Episode Detection

**During the datathon, ICU admission records were not initially available in `visit_occurrence`.** This meant we couldn't directly link measurements to distinct ICU stays — a patient with measurements spanning months appeared as one continuous record.

Before the fix, we inferred episode boundaries from gaps in measurement timestamps. A gap of >24 hours between consecutive ventilator measurements almost certainly represents a new episode or a gap between admissions. The code below shows this approach.

The datathon organisers added ICU admission records to `visit_occurrence` partway through. If your instance has `visit_occurrence` populated for ICU admissions, use that directly and skip the gap-detection logic.

We're keeping this here because the gap-detection approach is a useful fallback if visit data is absent or unreliable in your instance (see Pitfall 4). It's also a good illustration of how to think through a data structure problem when the expected table doesn't contain what you need.

> **First check:** Run the `inspect_table('visit_occurrence')` helper from Section 1 and look at a sample of rows — if ICU admissions are present, use that. If not, the gap-detection approach below is your fallback.

In [ ]:
# First: check if visit_occurrence has ICU admissions in your instance
# If it does, use that directly — this gap-detection approach is the fallback

query_visit_check = f"""
SELECT
  v.visit_concept_id,
  c.concept_name,
  COUNT(*) AS visit_count,
  COUNT(DISTINCT v.person_id) AS patient_count
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.visit_occurrence` v
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
  ON v.visit_concept_id = c.concept_id
GROUP BY v.visit_concept_id, c.concept_name
ORDER BY visit_count DESC
"""
visit_types = client.query(query_visit_check, job_config=job_config).to_dataframe()
print('Visit types in visit_occurrence:')
print(visit_types.to_string(index=False))
print('\nIf ICU admissions are present above, use visit_occurrence directly.')
print('If not — proceed with the gap-detection approach below.\n')

# ── Gap-detection fallback ────────────────────────────────────────────────────
# ← Replace with your validated invasive ventilation concept IDs
vent_concept_ids = []  # e.g. [FIO2_ID, PEEP_ID, TV_SET_ID]

if vent_concept_ids:
    ids_str = ','.join(str(i) for i in vent_concept_ids)
    query = f"""
    WITH vent_measurements AS (
      SELECT person_id, measurement_datetime
      FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
      WHERE measurement_concept_id IN ({ids_str})
    ),
    with_gaps AS (
      SELECT
        person_id,
        measurement_datetime,
        TIMESTAMP_DIFF(
          measurement_datetime,
          LAG(measurement_datetime) OVER (PARTITION BY person_id ORDER BY measurement_datetime),
          HOUR
        ) AS hours_since_last
      FROM vent_measurements
    ),
    episode_starts AS (
      SELECT
        person_id,
        measurement_datetime,
        CASE
          WHEN hours_since_last IS NULL OR hours_since_last > 24 THEN 1
          ELSE 0
        END AS is_new_episode
      FROM with_gaps
    ),
    episodes AS (
      SELECT
        person_id,
        measurement_datetime,
        SUM(is_new_episode) OVER (
          PARTITION BY person_id
          ORDER BY measurement_datetime
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS episode_number
      FROM episode_starts
    )
    SELECT
      person_id,
      episode_number,
      MIN(measurement_datetime) AS episode_start,
      MAX(measurement_datetime) AS episode_end,
      TIMESTAMP_DIFF(
        MAX(measurement_datetime),
        MIN(measurement_datetime),
        HOUR
      )                         AS episode_hours,
      COUNT(*)                  AS measurement_count
    FROM episodes
    GROUP BY person_id, episode_number
    HAVING episode_hours >= 48
    ORDER BY person_id, episode_number
    LIMIT 100
    """

    episodes = client.query(query, job_config=job_config).to_dataframe()
    print(f'Detected {len(episodes)} ventilation episodes ≥48h')
    print(f"Unique patients: {episodes['person_id'].nunique()}")
    print('\nEpisode duration summary (hours):')
    print(episodes['episode_hours'].describe())
    print('\nValidate against visit_occurrence for a sample patient before trusting these boundaries')
    episodes.head(20)
else:
    print('Set vent_concept_ids above to run the gap-detection fallback')

### 📝 Note: Year-Based Patterns Reflect Anonymisation, Not Data Gaps

AmsterdamUMCdb anonymises data by **binning dates** — year of birth and admission year are shifted or rounded to protect patient privacy. This means:

- Apparent 'gaps' or 'sparse years' in temporal data are likely **anonymisation artefacts**, not missing data
- Do not draw conclusions about data availability from year-level counts without checking the anonymisation documentation first
- Absolute years in the data may not correspond to actual calendar years

The query below is still useful for **understanding the temporal distribution of your data** — just interpret the output in light of anonymisation, not as a direct reflection of real-world data density.

> **Reference:** Check the [AmsterdamUMCdb anonymisation documentation](https://github.com/AmsterdamUMC/AmsterdamUMCdb) for the specific rules applied to your version.

In [ ]:
# Temporal distribution of a measurement — interpret with anonymisation in mind
# ← Replace with the concept ID to check
CHECK_CONCEPT_ID = None

if CHECK_CONCEPT_ID:
    query = f"""
    SELECT
      EXTRACT(YEAR FROM measurement_datetime) AS year,
      COUNT(*)                               AS measurement_count,
      COUNT(DISTINCT person_id)              AS patient_count
    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
    WHERE measurement_concept_id = {CHECK_CONCEPT_ID}
    GROUP BY year
    ORDER BY year
    """

    yearly = client.query(query, job_config=job_config).to_dataframe()
    print(f'Temporal distribution for concept {CHECK_CONCEPT_ID}:')
    print(yearly.to_string(index=False))
    print('\n⚠ Apparent year gaps may reflect anonymisation (date binning) rather than real data absence.')
    print('  Check anonymisation documentation before drawing conclusions about temporal patterns.')
else:
    print('Set CHECK_CONCEPT_ID above to run this check')

### 💡 Pearl: Two Ways to Query in Colab — Know the Difference

**Method 1 — Python (used throughout this notebook):**

```python
result = client.query(
    f"SELECT ... FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`",
    job_config=job_config
).to_dataframe()
```

✅ Parameterisable with f-strings  
✅ Works in loops  
✅ Required for any cohort-building query  

---

**Method 2 — BigQuery magic (quick one-off exploration only):**

```python
%%bigquery my_variable --project YOUR_PROJECT_ID
SELECT * FROM `amsterdamumcdb.van_gogh_2026_datathon.measurement` LIMIT 10
```

✅ Fast to type  
✅ Result automatically stored in `my_variable` as a DataFrame  
⚠ **Cannot use f-strings or Python variables** — all values must be hardcoded  
⚠ Not suitable for parameterised cohort queries  

---

> **Rule of thumb:** Use Method 2 only for quick one-off exploration. Switch to Method 1 the moment you need to parameterise anything or reuse the query.

---
## What's Next

You now have the tools to orient yourself in any AmsterdamUMCdb instance and find, validate, and work with concept IDs systematically.

The next modules build on this:

| Module | Topic |
|--------|-------|
| **Module 2** | Clinical Question to Cohort Workflow — applying these patterns to build a real cohort from a clinical hypothesis (fluid balance trajectories and weaning prediction) |
| **Module 3** | The BigTable Accumulation Pattern — how to build up a feature-rich table incrementally in BigQuery, the way the 2026 datathon team actually worked |

---

*ESICM LIVES26 Educational Series — Datathon Track*  
*Materials created from the 2026 ESICM INDICATE datathon experience*